# First Title
## Second Title
- item 1
- item 2
- item 3
**abcde

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
df=pd.read_csv('RFM.csv')

In [4]:
df.head()

,Date,InvoNo,CustomerKey,銷售金額
0,20180413,AP1804933,A1255,9997.02
1,20180423,AP1807826,A1188,9997.02
2,20180424,AP1800872,A0846,9997.02
3,20180426,AP1803170,A1204,9997.02
4,20180501,MA1806692,A0907,9997.02


In [17]:
import numpy as np
import pandas as pd

# ==========================================
# 1. 日期格式轉換與基準日設定
# ==========================================
# 將 'Date' 欄位轉為字串後，再以 pd.to_datetime 解析（相容文字如 '2018-01-01' 或整數如 20180101）
df['Date'] = pd.to_datetime(df['Date'].astype(str))

# 找出整個 sales 資料集中的最晚日期作為計算基準日
reference_date = df['Date'].max()

# ==========================================
# 2. 聚合計算 R, F, M
# ==========================================
df_rfm_agg = (
    df.groupby('CustomerKey')
    .agg(
        # Recency: 基準日 減去 該客戶最後一次消費日期的天數差
        Recency=('Date', lambda x: (reference_date - x.max()).days),
        # Frequency: 發票編號相異值計數
        Frequency=('InvoNo', 'nunique'),
        # Monetary: 銷售金額加總
        Monetary=('銷售金額', 'sum'),
    )
    .reset_index()
)

# ==========================================
# 3. 資料複製
# ==========================================
df_rfm = df_rfm_agg.copy()

# ==========================================
# 4. RFM 評分 (1-5 分，使用 pd.qcut)
# ==========================================
# 注意：
# - 使用 .rank(method='first') 可避免切分區間邊界重複 (Bin edges must be unique) 的錯誤
# - Recency: 天數越小代表越近消費，分數越高 (labels=[5, 4, 3, 2, 1])
# - Frequency & Monetary: 數值越大分數越高 (labels=[1, 2, 3, 4, 5])

df_rfm['R_Score'] = (
    pd.qcut(
        df_rfm['Recency'].rank(method='first'),
        q=5,
        labels=[5, 4, 3, 2, 1],
    )
    .astype(int)
)

df_rfm['F_Score'] = (
    pd.qcut(
        df_rfm['Frequency'].rank(method='first'),
        q=5,
        labels=[1, 2, 3, 4, 5],
    )
    .astype(int)
)

df_rfm['M_Score'] = (
    pd.qcut(
        df_rfm['Monetary'].rank(method='first'),
        q=5,
        labels=[1, 2, 3, 4, 5],
    )
    .astype(int)
)

# ==========================================
# 5. 客群標籤 (RFM_Segment) 劃分邏輯
# ==========================================
# 建立分群規則（順序由高優先權往低判定）：
conditions = [
    # 高價值核心客戶：近期有買、常買、花得多
    (df_rfm['R_Score'] >= 4) & (df_rfm['F_Score'] >= 4) & (df_rfm['M_Score'] >= 4),
    
    # 忠誠客戶：近期活躍且頻次高
    (df_rfm['R_Score'] >= 3) & (df_rfm['F_Score'] >= 3),
    
    # 潛力客戶 / 新客：近期有消費，但累積頻次尚低
    (df_rfm['R_Score'] >= 4) & (df_rfm['F_Score'] <= 2),
    
    # 重要留存客戶 (流失風險高)：過去常買或貢獻高，但近期很久沒來
    (df_rfm['R_Score'] <= 2) & ((df_rfm['F_Score'] >= 3) | (df_rfm['M_Score'] >= 4)),
    
    # 流失 / 沉睡客戶：很久沒買且以前頻次也不高
    (df_rfm['R_Score'] <= 2) & (df_rfm['F_Score'] <= 2),
]

segment_names = [
    '高價值客戶 (Champions)',
    '忠誠客戶 (Loyal Customers)',
    '潛力客戶 (Potential)',
    '重要留存客戶 (At Risk)',
    '流失/沉睡客戶 (Lost)',
]

# 套用條件標籤，未吻合者歸類為一般客戶
df_rfm['RFM_Segment'] = np.select(conditions, segment_names, default='一般維護客戶 (Others)')



import pandas as pd

# 1. 調整 Pandas 全域設定：確保 print 時一定會折疊並顯示欄列數資訊
pd.set_option('display.max_rows', 10)       # 超過10列就折疊
pd.set_option('display.show_dimensions', True) # 關鍵：強制 print 時一定要印出 [X rows x Y columns]

# 2. 抽取核心欄位並複製
result_df = df_rfm[['CustomerKey', 'Frequency', 'Monetary', 'Recency']].copy()

# 3. 修改欄位名稱完全對齊老師
result_df.columns = ['Customer', 'F', 'M', 'R']

# 4. 確保 F 和 R 顯示為整數
result_df['F'] = result_df['F'].astype(int)
result_df['R'] = result_df['R'].astype(int)

# 5. 印出結果 (這樣就會包含中間的 ...、末尾的 727-731 以及最底下的行列數提示)
print(result_df)


print()
import pandas as pd

# 1. 調整環境設定（確保 8 個欄位能橫向併排，不折行、不漏字）
pd.set_option('display.max_rows', 10)          # 超過 10 列就自動折疊中間
pd.set_option('display.max_columns', None)     # 完整顯示所有欄位
pd.set_option('display.width', 1000)           # 拓寬字元限制，防止表格斷行
pd.set_option('display.show_dimensions', True) # 顯示最底下的 [732 rows x 8 columns]

# 2. 只印出你原本算好的完整資料表
print(df_rfm)


    Customer   F            M   R
0      A0350  25  51238.11440   9
1      A0351   6  17890.31200  16
2      A0352  14  46424.03852  14
3      A0353  14  27607.68636  24
4      A0354   6   9973.92064  31
..       ...  ..          ...  ..
727    A1413  23  52738.79456  10
728    A1415  20  53019.49664  17
729    A1416  24  49938.38378   4
730    A1419  22  64257.84520  50
731    A1420  14  25054.18342  85

[732 rows x 4 columns]

    CustomerKey  Recency  Frequency     Monetary  R_Score  F_Score  M_Score             RFM_Segment
0         A0350        9         25  51238.11440        5        5        4       高價值客戶 (Champions)
1         A0351       16          6  17890.31200        4        2        3        潛力客戶 (Potential)
2         A0352       14         14  46424.03852        4        3        4  忠誠客戶 (Loyal Customers)
3         A0353       24         14  27607.68636        3        3        3  忠誠客戶 (Loyal Customers)
4         A0354       31          6   9973.92064        3        2 

In [22]:
# 查看各客群的客戶數佔比與平均指標
segment_summary = df_rfm.groupby('RFM_Segment').agg(
    客戶數=('CustomerKey', 'count'),
    平均Recency=('Recency', 'mean'),
    平均Frequency=('Frequency', 'mean'),
    平均Monetary=('Monetary', 'mean')
).reset_index()

# 關鍵：使用 style 來做真正的網頁級置中對齊，並四捨五入小數點後 2 位（看起來更乾淨）
segment_summary.style.set_properties(**{'text-align': 'center'}).set_table_styles([
    {'selector': 'th', 'props': [('text-align', 'center')]}
]).format({
    '平均Recency': '{:.2f}',
    '平均Frequency': '{:.2f}',
    '平均Monetary': '{:.2f}'
})

,RFM_Segment,客戶數,平均Recency,平均Frequency,平均Monetary
0,一般維護客戶 (Others),44,26.05,4.48,11702.51
1,忠誠客戶 (Loyal Customers),183,16.64,16.02,40341.82
2,流失/沉睡客戶 (Lost),201,108.76,4.01,9243.18
3,潛力客戶 (Potential),44,10.16,4.61,12515.68
4,重要留存客戶 (At Risk),92,64.23,13.32,36320.53
5,高價值客戶 (Champions),168,8.43,23.92,61183.75
